Cell 1: Setup and model parameters

In [11]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROCESSED = ROOT / "data" / "processed"
INTERIM = ROOT / "data" / "interim"
if not PROCESSED.exists():
    ROOT = Path(r"C:\Users\ASUS\ipl-business-intelligence")
    PROCESSED = ROOT / "data" / "processed"
    INTERIM = ROOT / "data" / "interim"

master = pd.read_csv(PROCESSED / "player_season_master.csv")
stats = pd.read_csv(INTERIM / "player_season_stats.csv")          # every player, every season
matches_clean = pd.read_csv(INTERIM / "matches_clean.csv")

# ------------------- MODEL PARAMETERS: every assumption lives here -------------------
V = 20                 # runs one wicket is worth (a bowler gains it, a batter loses it when dismissed)
ON_SHARE = 0.70        # share of business value created on the field; the rest is off-field brand value
STAR_W = 10            # a six or a wicket counts as this many extra balls of screen time
VALUE_MULTIPLE = 1.0   # business value per rupee of salary at the market average (1.0 = fair market)
TEAM_BRAND = {}        # optional brand weights, e.g. {"CSK": 1.3, "MI": 1.3}; unlisted teams get 1.0
AUCTION_ONLY = True    # same rule as the analysis notebook

print("Master rows:", len(master), "| league-wide player-season rows:", len(stats))
print("Seasons in stats:", int(stats["season_year"].min()), "-", int(stats["season_year"].max()))

Master rows: 514 | league-wide player-season rows: 2938
Seasons in stats: 2008 - 2024


Cell 2: League averages per season

In [12]:
par = stats.groupby("season_year").agg(
    runs=("runs", "sum"), faced=("balls_faced", "sum"), dism=("dismissals", "sum"),
    conceded=("runs_conceded", "sum"), bowled=("balls_bowled", "sum"), wkts=("wickets", "sum"),
)
par["rpb"] = par["runs"] / par["faced"]        # runs per ball faced
par["dpb"] = par["dism"] / par["faced"]        # dismissals per ball faced
par["cpb"] = par["conceded"] / par["bowled"]   # runs conceded per legal ball
par["wpb"] = par["wkts"] / par["bowled"]       # bowler wickets per legal ball
par["strike_rate"] = (par["rpb"] * 100).round(1)
par["economy"] = (par["cpb"] * 6).round(2)
print(par.loc[2020:, ["strike_rate", "economy", "dpb", "wpb"]].round(4).to_string())

             strike_rate  economy     dpb     wpb
season_year                                      
2020               131.6     8.16  0.0480  0.0450
2021               126.9     7.92  0.0513  0.0482
2022               133.9     8.39  0.0530  0.0495
2023               141.7     8.84  0.0531  0.0500
2024               150.6     9.41  0.0539  0.0509


Cell 3: Impact score, with a whole-league sanity check

In [13]:
def add_impact(df, season_col, wicket_value):
    """Batting and bowling impact in 'runs above average'. Both parts sum to zero across the whole league."""
    out = df.copy()
    rpb = out[season_col].map(par["rpb"])
    dpb = out[season_col].map(par["dpb"])
    cpb = out[season_col].map(par["cpb"])
    wpb = out[season_col].map(par["wpb"])
    out["bat_impact"] = (out["runs"] - wicket_value * out["dismissals"]
                         - out["balls_faced"] * (rpb - wicket_value * dpb))
    out["bowl_impact"] = (out["balls_bowled"] * cpb - out["runs_conceded"]
                          + wicket_value * (out["wickets"] - out["balls_bowled"] * wpb))
    out["impact"] = out["bat_impact"] + out["bowl_impact"]
    return out


# Sanity check on the WHOLE league: the sums must be zero, and the top names should look familiar
league = add_impact(stats, "season_year", V)
print("Sum of impact per season (must be about 0):")
print(league.groupby("season_year")[["bat_impact", "bowl_impact"]].sum().round(2).tail(3).to_string())

recent = league[league["season_year"].between(2022, 2024)]
top = (recent.sort_values("impact", ascending=False).groupby("season_year").head(5)
             .sort_values(["season_year", "impact"], ascending=[True, False]))
print("\nTop 5 by impact each season (whole league):")
print(top[["season_year", "player", "runs", "wickets", "impact"]].round(0).to_string(index=False))

Sum of impact per season (must be about 0):
             bat_impact  bowl_impact
season_year                         
2022               -0.0          0.0
2023               -0.0          0.0
2024               -0.0          0.0

Top 5 by impact each season (whole league):
 season_year       player  runs  wickets  impact
        2022   JC Buttler 863.0      0.0   401.0
        2022     KL Rahul 616.0      0.0   249.0
        2022    DA Miller 481.0      0.0   247.0
        2022   AD Russell 335.0     17.0   232.0
        2022    HH Pandya 487.0      8.0   176.0
        2023 Shubman Gill 890.0      0.0   390.0
        2023    MM Sharma   0.0     27.0   304.0
        2023 F du Plessis 730.0      0.0   302.0
        2023    DP Conway 672.0      0.0   242.0
        2023      V Kohli 639.0      0.0   237.0
        2024      V Kohli 741.0      0.0   296.0
        2024    SP Narine 488.0     17.0   246.0
        2024    JJ Bumrah  12.0     20.0   242.0
        2024     N Pooran 499.0      0.0

Cell 4: Business value and ROI

In [14]:
def value_players(df, wicket_value, on_share, star_w, multiple, team_brand, season_col="season"):
    out = add_impact(df, season_col, wicket_value)
    out["visibility"] = out["balls_faced"] + out["balls_bowled"] + star_w * (out["sixes"] + out["wickets"])
    brand = out["team"].map(team_brand or {}).fillna(1.0)

    pool = out["price_cr"].sum() * multiple            # total business value to share out
    pool_on, pool_off = pool * on_share, pool * (1 - on_share)

    positive = out["impact"].clip(lower=0)             # a below-average player creates no on-field value (never negative)
    cr_per_run = pool_on / positive.sum()
    weighted_vis = out["visibility"] * brand
    cr_per_vis = pool_off / weighted_vis.sum()

    out["value_on_cr"] = positive * cr_per_run         # on-field value: crore per run above average
    out["value_off_cr"] = weighted_vis * cr_per_vis    # off-field value: brand value from screen time and star moments
    out["value_cr"] = out["value_on_cr"] + out["value_off_cr"]
    out["roi"] = out["value_cr"] / out["price_cr"]
    out["surplus_cr"] = out["value_cr"] - out["price_cr"]
    return out, {"cr_per_run": cr_per_run, "cr_per_vis_unit": cr_per_vis, "pool_cr": pool}


df = master[master["acquisition"] == "sold"].copy() if AUCTION_ONLY else master.copy()
df["season"] = df["season"].astype(int)

model, rates = value_players(df, V, ON_SHARE, STAR_W, VALUE_MULTIPLE, TEAM_BRAND)

# availability = share of the team's matches the player appeared in
tm = pd.concat([
    matches_clean[["season_year", "team1"]].rename(columns={"team1": "team"}),
    matches_clean[["season_year", "team2"]].rename(columns={"team2": "team"}),
])
team_matches = tm.groupby(["season_year", "team"]).size().rename("team_matches").reset_index()
model = (model.merge(team_matches, left_on=["season", "team"], right_on=["season_year", "team"], how="left")
              .drop(columns="season_year"))
model["availability"] = (model["matches_played"] / model["team_matches"]).clip(upper=1).round(2)

print(f"Money pool shared out: {rates['pool_cr']:.1f} crore  (= total auction spend x {VALUE_MULTIPLE})")
print(f"Market-implied value of 1 run above average: {rates['cr_per_run'] * 100:.2f} lakh")
print(f"Value of 1 unit of screen time: {rates['cr_per_vis_unit'] * 100:.3f} lakh")
print(f"\nOverall ROI (total value / total price): {model['value_cr'].sum() / model['price_cr'].sum():.2f}  (equals VALUE_MULTIPLE by construction)")
print(f"Purchases with ROI above 1: {(model['roi'] > 1).mean():.0%}")
print(f"Purchases with zero on-field value (impact of 0 or less): {(model['impact'] <= 0).sum()}")

Money pool shared out: 949.1 crore  (= total auction spend x 1.0)
Market-implied value of 1 run above average: 12.06 lakh
Value of 1 unit of screen time: 0.497 lakh

Overall ROI (total value / total price): 1.00  (equals VALUE_MULTIPLE by construction)
Purchases with ROI above 1: 28%
Purchases with zero on-field value (impact of 0 or less): 269


Cell 5: Best and worst purchases

In [15]:
cols = ["season", "player_name", "team", "role", "price_cr", "matches_played", "availability",
        "impact", "value_cr", "roi", "surplus_cr"]
reg = model[model["matches_played"] >= 5]

print("HIGHEST ROI (5+ matches, price 1 crore or more):")
print(reg[reg["price_cr"] >= 1].sort_values("roi", ascending=False).head(12)[cols].round(2).to_string(index=False))

print("\nBIGGEST SURPLUS (value created above the price paid):")
print(model.sort_values("surplus_cr", ascending=False).head(12)[cols].round(2).to_string(index=False))

print("\nBIGGEST SHORTFALL (price paid above value created, price 5 crore or more):")
print(model[model["price_cr"] >= 5].sort_values("surplus_cr").head(12)[cols].round(2).to_string(index=False))

HIGHEST ROI (5+ matches, price 1 crore or more):
 season       player_name team         role  price_cr  matches_played  availability  impact  value_cr   roi  surplus_cr
   2022      Devon Conway  CSK       Batter      1.00             7.0          0.50   83.63     11.54 11.54       10.54
   2022      David Miller   GT       Batter      3.00            16.0          1.00  246.77     32.58 10.86       29.58
   2022     Kuldeep Yadav   DC       Bowler      2.00            14.0          1.00  136.19     19.31  9.65       17.31
   2022    N. Tilak Varma   MI  All-Rounder      1.70            14.0          1.00   84.18     12.51  7.36       10.81
   2022     Aiden Markram  SRH       Batter      2.60            13.0          0.93  115.09     16.41  6.31       13.81
   2024 Mustafizur Rahman  CSK       Bowler      2.00             9.0          0.64   76.97     11.01  5.50        9.01
   2022       Umesh Yadav  KKR       Bowler      2.00            12.0          0.86   62.46     10.16  5.08    

Cell 6: Franchise Efficiency Score

In [16]:
team_tbl = model.groupby("team").agg(
    purchases=("player_name", "count"),
    spend_cr=("price_cr", "sum"),
    value_cr=("value_cr", "sum"),
    played_pct=("played", lambda s: round(s.mean() * 100)),
    avg_availability=("availability", "mean"),
)
unused = model[~model["played"]].groupby("team")["price_cr"].sum().rename("unused_cr")
team_tbl = team_tbl.join(unused).fillna({"unused_cr": 0})
team_tbl["unused_pct"] = (team_tbl["unused_cr"] / team_tbl["spend_cr"] * 100).round(1)
team_tbl["efficiency"] = (team_tbl["value_cr"] / team_tbl["spend_cr"]).round(2)
team_tbl = team_tbl.sort_values("efficiency", ascending=False)
print(team_tbl.round(2).to_string())

print("\nEfficiency by season (value / spend):")
ts = model.groupby(["team", "season"]).agg(spend=("price_cr", "sum"), value=("value_cr", "sum"))
ts["efficiency"] = ts["value"] / ts["spend"]
print(ts["efficiency"].unstack().round(2).to_string())

fig = px.bar(team_tbl.reset_index(), x="team", y="efficiency", color="unused_pct",
             title="Franchise Efficiency Score (business value / auction spend)",
             labels={"efficiency": "Efficiency (1.0 = market average)", "unused_pct": "Unused spend %"})
fig.add_hline(y=1.0, line_dash="dash")
fig.show()

      purchases  spend_cr  value_cr  played_pct  avg_availability  unused_cr  unused_pct  efficiency
team                                                                                                
DC           34     81.45    116.63          76              0.38       6.30         7.7        1.43
GT           35     96.95    113.92          66              0.33      21.30        22.0        1.18
MI           37     85.10     98.37          76              0.36      15.40        18.1        1.16
LSG          34     91.00    105.70          68              0.34      18.95        20.8        1.16
RCB          32     82.85     92.64          62              0.32       9.10        11.0        1.12
SRH          39    134.40    131.36          74              0.39       4.95         3.7        0.98
PBKS         37    113.50    100.84          59              0.35       7.85         6.9        0.89
RR           35     85.20     71.47          71              0.32       2.40         2.8   

Cell 7: Value against price

In [17]:
plot_df = model[model["played"]]
fig = px.scatter(plot_df, x="price_cr", y="value_cr", color="role", hover_name="player_name",
                 hover_data=["season", "team", "matches_played", "impact", "roi"],
                 title="Business value created vs price paid (players who played)",
                 labels={"price_cr": "Price (crore)", "value_cr": "Estimated business value (crore)"})
top_val = max(plot_df["price_cr"].max(), plot_df["value_cr"].max())
fig.add_shape(type="line", x0=0, y0=0, x1=top_val, y1=top_val, line=dict(dash="dash"))
fig.show()

Cell 8: Sensitivity, the most important cell

In [18]:
scenarios = {
    "baseline": {},
    "wicket = 10 runs": {"V": 10},
    "wicket = 30 runs": {"V": 30},
    "on-field share 50%": {"on_share": 0.50},
    "on-field share 90%": {"on_share": 0.90},
    "no star bonus": {"star_w": 0},
}
base_roi = model["roi"].values
base_top = set(map(tuple, model.sort_values("surplus_cr", ascending=False).head(15)[["season", "player_name"]].values))
base_team = team_tbl["efficiency"]
mask = model["played"].values

rows = []
for name, kw in scenarios.items():
    p = {"V": V, "on_share": ON_SHARE, "star_w": STAR_W}
    p.update(kw)
    m, _ = value_players(df, p["V"], p["on_share"], p["star_w"], VALUE_MULTIPLE, TEAM_BRAND)
    top15 = set(map(tuple, m.sort_values("surplus_cr", ascending=False).head(15)[["season", "player_name"]].values))
    g = m.groupby("team")[["value_cr", "price_cr"]].sum()
    eff = g["value_cr"] / g["price_cr"]
    rows.append({
        "scenario": name,
        "roi_rank_corr": round(pd.Series(m["roi"].values[mask]).corr(pd.Series(base_roi[mask]), method="spearman"), 2),
        "top15_surplus_overlap": len(base_top & top15),
        "team_rank_corr": round(eff.corr(base_team, method="spearman"), 2),
        "best_team": eff.idxmax(),
        "worst_team": eff.idxmin(),
    })
print(pd.DataFrame(rows).to_string(index=False))
print("\nHigh correlations and large overlaps mean the conclusions do not depend on the assumption.")


          scenario  roi_rank_corr  top15_surplus_overlap  team_rank_corr best_team worst_team
          baseline           1.00                     15            1.00        DC        KKR
  wicket = 10 runs           0.98                     12            0.91        DC        KKR
  wicket = 30 runs           0.99                     14            1.00        DC        KKR
on-field share 50%           0.99                     14            0.96        DC        KKR
on-field share 90%           0.98                     15            0.98        DC        KKR
     no star bonus           0.99                     14            1.00        DC        KKR

High correlations and large overlaps mean the conclusions do not depend on the assumption.


Cell 9: Player ROI Calculator (the logic for the app)


In [19]:
def business_roi(price_cr, season=2024, team=None, runs=0, balls_faced=0, dismissals=0,
                 wickets=0, balls_bowled=0, runs_conceded=0, sixes=0):
    p = par.loc[season]
    bat = runs - V * dismissals - balls_faced * (p["rpb"] - V * p["dpb"])
    bowl = balls_bowled * p["cpb"] - runs_conceded + V * (wickets - balls_bowled * p["wpb"])
    impact = bat + bowl
    visibility = balls_faced + balls_bowled + STAR_W * (sixes + wickets)
    on = max(impact, 0) * rates["cr_per_run"]
    off = visibility * TEAM_BRAND.get(team, 1.0) * rates["cr_per_vis_unit"]
    return {"impact_runs": int(round(impact)), "on_field_value_cr": round(float(on), 2),
            "off_field_value_cr": round(float(off), 2), "total_value_cr": round(float(on + off), 2),
            "roi": round(float((on + off) / price_cr), 2), "surplus_cr": round(float(on + off - price_cr), 2)}


print("15 cr opener, 450 runs off 300 balls, out 12 times, 25 sixes:")
print(business_roi(15, runs=450, balls_faced=300, dismissals=12, sixes=25))
print("\n12 cr fast bowler, 20 wickets, 64 overs, 500 runs conceded:")
print(business_roi(12, wickets=20, balls_bowled=384, runs_conceded=500))
print("\n0.2 cr bench player who never played:")
print(business_roi(0.2))

15 cr opener, 450 runs off 300 balls, out 12 times, 25 sixes:
{'impact_runs': 82, 'on_field_value_cr': 9.86, 'off_field_value_cr': 2.74, 'total_value_cr': 12.6, 'roi': 0.84, 'surplus_cr': -2.4}

12 cr fast bowler, 20 wickets, 64 overs, 500 runs conceded:
{'impact_runs': 111, 'on_field_value_cr': 13.43, 'off_field_value_cr': 2.9, 'total_value_cr': 16.34, 'roi': 1.36, 'surplus_cr': 4.34}

0.2 cr bench player who never played:
{'impact_runs': 0, 'on_field_value_cr': 0.0, 'off_field_value_cr': 0.0, 'total_value_cr': 0.0, 'roi': 0.0, 'surplus_cr': -0.2}


Cell 10: Save outputs for the app

In [20]:
model.to_csv(PROCESSED / "player_roi.csv", index=False)
team_tbl.to_csv(PROCESSED / "team_efficiency.csv")

params = {
    "wicket_value_runs": V, "on_field_share": ON_SHARE, "star_weight": STAR_W,
    "value_multiple": VALUE_MULTIPLE, "team_brand": TEAM_BRAND, "auction_only": AUCTION_ONLY,
    "rates": {k: float(v) for k, v in rates.items()},
    "par": par[["rpb", "dpb", "cpb", "wpb"]].round(6).reset_index().to_dict(orient="records"),
}
with open(PROCESSED / "roi_params.json", "w") as f:
    json.dump(params, f, indent=2)

print("Saved player_roi.csv, team_efficiency.csv and roi_params.json to", PROCESSED)

Saved player_roi.csv, team_efficiency.csv and roi_params.json to c:\Users\ASUS\ipl-business-intelligence\data\processed


In [21]:
# how concentrated is the value? (players)
v = model["value_cr"].sort_values(ascending=False)
n10 = int(len(v) * 0.10)
print(f"Top 10% of purchases ({n10} players) create {v.head(n10).sum() / v.sum():.0%} of all value")
print(f"Purchases needed to reach half of all value: {int((v.cumsum() / v.sum() < 0.5).sum() + 1)} of {len(v)}")
print("\nROI of purchases who played:")
print(model.loc[model["played"], "roi"].describe(percentiles=[.25, .5, .75, .9]).round(2).to_string())

# how much does each team's score depend on ONE player?
def team_eff(d):
    g = d.groupby("team")[["value_cr", "price_cr"]].sum()
    return g["value_cr"] / g["price_cr"]

base = team_eff(model)
rows = []
for team, g in model.groupby("team"):
    best = g.loc[g["value_cr"].idxmax()]
    rest = g.drop(g["value_cr"].idxmax())
    rows.append({
        "team": team,
        "efficiency": round(base[team], 2),
        "top_player": f"{best['player_name']} ({int(best['season'])})",
        "top_player_share_pct": round(best["value_cr"] / g["value_cr"].sum() * 100),
        "efficiency_without_top": round(rest["value_cr"].sum() / rest["price_cr"].sum(), 2),
    })
conc = pd.DataFrame(rows).set_index("team").sort_values("efficiency", ascending=False)
conc["rank_with"] = conc["efficiency"].rank(ascending=False, method="min").astype(int)
conc["rank_without"] = conc["efficiency_without_top"].rank(ascending=False, method="min").astype(int)
print("\nTeam score with and without its single most valuable purchase:")
print(conc.to_string())

Top 10% of purchases (35 players) create 63% of all value
Purchases needed to reach half of all value: 24 of 356

ROI of purchases who played:
count    242.00
mean       3.99
std       11.68
min        0.00
25%        0.26
50%        0.67
75%        2.32
90%        9.45
max      106.97

Team score with and without its single most valuable purchase:
      efficiency               top_player  top_player_share_pct  efficiency_without_top  rank_with  rank_without
team                                                                                                            
DC          1.43    Tristan Stubbs (2024)                    23                    1.10          1             1
GT          1.18      Mohit Sharma (2023)                    35                    0.77          2             6
MI          1.16     Piyush Chawla (2023)                    15                    0.99          3             2
LSG         1.16       Mohsin Khan (2022)                    20                    0